#### Importing Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (12, 6)

#### Loading the Dataset

In [3]:
data_path = Path("../data/raw/Online Retail.xlsx")

df = pd.read_excel(data_path, engine="openpyxl")

print(f"Dataset Shape: {df.shape}")
print(f"Date Range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")

display(df.head())

Dataset Shape: (541909, 8)
Date Range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,"17,850.00",United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,"17,850.00",United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom


#### Saving the Raw Dataset in Parquet Format

In [6]:
parquet_path = Path("../data/processed/online_retail_raw.parquet")
parquet_path.parent.mkdir(parents=True, exist_ok=True)

text_columns = ["InvoiceNo", "StockCode", "Description", "Country"]

df[text_columns] = df[text_columns].astype("string")

df.to_parquet(parquet_path, engine="pyarrow", index=False)

print(f"Parquet file saved to: {parquet_path}")
print(f"File size: {parquet_path.stat().st_size / (1024 ** 2):.2f} MB")

print("\nDataset Shape:")
print(df.shape)

print("\nColumn Data Types:")
display(df.dtypes.to_frame("Data Type"))

print("\nMissing Values:")
display(df.isna().sum().to_frame("Missing Values"))

Parquet file saved to: ..\data\processed\online_retail_raw.parquet
File size: 3.42 MB

Dataset Shape:
(541909, 8)

Column Data Types:


,Data Type
InvoiceNo,string
StockCode,string
Description,string
Quantity,int64
InvoiceDate,datetime64[us]
UnitPrice,float64
CustomerID,float64
Country,string



Missing Values:


,Missing Values
InvoiceNo,0
StockCode,0
Description,1454
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,135080
Country,0


#### Creating Transaction-Level Revenue Variables

In [7]:
eda_df = df.copy()

eda_df["Transaction_Date"] = eda_df["InvoiceDate"].dt.normalize()
eda_df["Line_Amount"] = eda_df["Quantity"] * eda_df["UnitPrice"]
eda_df["Is_Cancellation"] = eda_df["InvoiceNo"].str.startswith("C")

eda_df["Transaction_Type"] = np.select(
    [
        eda_df["Is_Cancellation"],
        eda_df["Quantity"] < 0,
        eda_df["UnitPrice"] < 0,
        eda_df["UnitPrice"] == 0
    ],
    [
        "Cancellation",
        "Other Negative Quantity",
        "Negative Unit Price",
        "Zero Unit Price"
    ],
    default="Positive Quantity and Price"
)

display(
    eda_df[
        [
            "InvoiceNo",
            "StockCode",
            "Transaction_Date",
            "Quantity",
            "UnitPrice",
            "Line_Amount",
            "Is_Cancellation",
            "Transaction_Type"
        ]
    ].head(10)
)

print("\nTransaction Type Distribution:")
display(
    eda_df["Transaction_Type"]
    .value_counts()
    .rename_axis("Transaction_Type")
    .reset_index(name="Records")
)

,InvoiceNo,StockCode,Transaction_Date,Quantity,UnitPrice,Line_Amount,Is_Cancellation,Transaction_Type
0,536365,85123A,2010-12-01,6,2.55,15.30,False,Positive Quantity and Price
1,536365,71053,2010-12-01,6,3.39,20.34,False,Positive Quantity and Price
2,536365,84406B,2010-12-01,8,2.75,22.00,False,Positive Quantity and Price
3,536365,84029G,2010-12-01,6,3.39,20.34,False,Positive Quantity and Price
4,536365,84029E,2010-12-01,6,3.39,20.34,False,Positive Quantity and Price
5,536365,22752,2010-12-01,2,7.65,15.30,False,Positive Quantity and Price
6,536365,21730,2010-12-01,6,4.25,25.50,False,Positive Quantity and Price
7,536366,22633,2010-12-01,6,1.85,11.10,False,Positive Quantity and Price
8,536366,22632,2010-12-01,6,1.85,11.10,False,Positive Quantity and Price
9,536367,84879,2010-12-01,32,1.69,54.08,False,Positive Quantity and Price



Transaction Type Distribution:


,Transaction_Type,Records
0,Positive Quantity and Price,530104
1,Cancellation,9288
2,Other Negative Quantity,1336
3,Zero Unit Price,1179
4,Negative Unit Price,2


**Key Observations**
- **Positive-quantity, positive-price transactions account for 530,104 records (97.82%)**, confirming that conventional sales transactions dominate the dataset.
- **Cancellation invoices account for 9,288 records (1.71%)**, highlighting the importance of distinguishing returns and cancellations from completed sales when calculating revenue.
- **An additional 1,336 records contain negative quantities outside cancellation invoices**, while 1,179 records have zero unit prices and only 2 contain negative unit prices. These entries require separate treatment during revenue construction.
- **Transaction classification provides a foundation for distinguishing merchandise sales from financial adjustments**, but transaction type alone cannot identify all non-merchandise entries, such as postage, fees, and discounts.

**EDA Decision**

Retain all transaction categories during exploratory analysis and compare their financial contributions before establishing the final daily merchandise revenue definition.

#### Analyzing Financial Contributions by Transaction Type

In [8]:
transaction_financial_summary = (
    eda_df.groupby("Transaction_Type")
    .agg(
        Transaction_Records=("Line_Amount", "size"),
        Total_Amount=("Line_Amount", "sum"),
        Average_Line_Amount=("Line_Amount", "mean"),
        Median_Line_Amount=("Line_Amount", "median")
    )
    .reset_index()
)

transaction_financial_summary["Record_Percentage"] = (
    transaction_financial_summary["Transaction_Records"]
    / len(eda_df) * 100
)

transaction_financial_summary["Contribution_to_Net_Percentage"] = (
    transaction_financial_summary["Total_Amount"]
    / eda_df["Line_Amount"].sum() * 100
)

transaction_financial_summary = transaction_financial_summary[
    [
        "Transaction_Type",
        "Transaction_Records",
        "Record_Percentage",
        "Total_Amount",
        "Contribution_to_Net_Percentage",
        "Average_Line_Amount",
        "Median_Line_Amount"
    ]
].sort_values("Total_Amount", ascending=False)

display(transaction_financial_summary)

print(
    f"\nPreliminary Net Transaction Amount: "
    f"{eda_df['Line_Amount'].sum():,.2f}"
)

,Transaction_Type,Transaction_Records,Record_Percentage,Total_Amount,Contribution_to_Net_Percentage,Average_Line_Amount,Median_Line_Amount
3,Positive Quantity and Price,530104,97.82,"10,666,684.54",109.43,20.12,9.90
4,Zero Unit Price,1179,0.22,0.00,0.00,0.00,0.00
2,Other Negative Quantity,1336,0.25,0.00,0.00,0.00,-0.00
1,Negative Unit Price,2,0.00,"-22,124.12",-0.23,"-11,062.06","-11,062.06"
0,Cancellation,9288,1.71,"-896,812.49",-9.20,-96.56,-8.50



Preliminary Net Transaction Amount: 9,747,747.93


**Key Observations**
- **Positive-quantity, positive-price transactions generate 10,666,684.54, representing 109.43% of preliminary net transaction value**, demonstrating that positive financial activity exceeds the final net amount before deductions.
- **Cancellation invoices reduce the preliminary net amount by 896,812.49 (9.20%)**, making them the largest negative financial component and an important consideration when defining gross versus net merchandise revenue.
- **Two negative-price adjustments contribute an additional reduction of 22,124.12 (0.23%)**, while zero-price transactions and other negative-quantity records have no direct monetary contribution.
- **The preliminary net transaction amount totals 9,747,747.93**, reflecting positive transactions after cancellations and negative-price adjustments. This figure is not yet a validated merchandise revenue total because it includes non-merchandise financial entries.

**EDA Decision**

Examine non-merchandise stock codes and their financial contributions before selecting the daily revenue measure. Maintain a clear distinction between gross merchandise sales, merchandise returns, and accounting adjustments to support consistent forecasting.

#### Analyzing Non-Merchandise Transaction Contributions

In [9]:
non_merchandise_codes = [
    "POST", "DOT", "C2", "M", "D", "S",
    "BANK CHARGES", "AMAZONFEE", "CRUK", "B"
]

non_merchandise_df = eda_df[
    eda_df["StockCode"].isin(non_merchandise_codes)
].copy()

non_merchandise_summary = (
    non_merchandise_df.groupby("StockCode")
    .agg(
        Transaction_Records=("InvoiceNo", "size"),
        Unique_Invoices=("InvoiceNo", "nunique"),
        Positive_Amount=("Line_Amount", lambda x: x[x > 0].sum()),
        Negative_Amount=("Line_Amount", lambda x: x[x < 0].sum()),
        Net_Amount=("Line_Amount", "sum")
    )
    .sort_values("Net_Amount", ascending=False)
)

display(non_merchandise_summary)

print("\nOverall Non-Merchandise Summary:")
print(f"Transaction Records: {len(non_merchandise_df):,}")
print(f"Positive Amount: {non_merchandise_df.loc[non_merchandise_df['Line_Amount'] > 0, 'Line_Amount'].sum():,.2f}")
print(f"Negative Amount: {non_merchandise_df.loc[non_merchandise_df['Line_Amount'] < 0, 'Line_Amount'].sum():,.2f}")
print(f"Net Amount: {non_merchandise_df['Line_Amount'].sum():,.2f}")

print("\nShare of Preliminary Net Transaction Amount:")
print(
    f"{non_merchandise_df['Line_Amount'].sum() / eda_df['Line_Amount'].sum() * 100:.2f}%"
)

,Transaction_Records,Unique_Invoices,Positive_Amount,Negative_Amount,Net_Amount
StockCode,,,,,
DOT,710,710,"206,248.77",-3.29,"206,245.48"
POST,1256,1254,"78,101.88","-11,871.24","66,230.64"
C2,144,144,"7,051.00",-65.00,"6,986.00"
S,63,19,63.05,"-3,112.44","-3,049.39"
D,77,65,0.00,"-5,696.22","-5,696.22"
BANK CHARGES,37,36,165.00,"-7,340.64","-7,175.64"
CRUK,16,16,0.00,"-7,933.43","-7,933.43"
B,3,3,"11,062.06","-22,124.12","-11,062.06"
M,571,517,"78,110.27","-146,784.46","-68,674.19"



Overall Non-Merchandise Summary:
Transaction Records: 2,911
Positive Amount: 394,563.12
Negative Amount: -440,212.43
Net Amount: -45,649.31

Share of Preliminary Net Transaction Amount:
-0.47%


In [10]:
display(non_merchandise_summary)

,Transaction_Records,Unique_Invoices,Positive_Amount,Negative_Amount,Net_Amount
StockCode,,,,,
DOT,710,710,"206,248.77",-3.29,"206,245.48"
POST,1256,1254,"78,101.88","-11,871.24","66,230.64"
C2,144,144,"7,051.00",-65.00,"6,986.00"
S,63,19,63.05,"-3,112.44","-3,049.39"
D,77,65,0.00,"-5,696.22","-5,696.22"
BANK CHARGES,37,36,165.00,"-7,340.64","-7,175.64"
CRUK,16,16,0.00,"-7,933.43","-7,933.43"
B,3,3,"11,062.06","-22,124.12","-11,062.06"
M,571,517,"78,110.27","-146,784.46","-68,674.19"


**Key Observations**
- **The ten identified non-merchandise stock codes account for 2,911 transaction records**, generating 394,563.12 in positive amounts and -440,212.43 in negative amounts, resulting in a net contribution of -45,649.31 (-0.47% of preliminary net transaction value).
- **DOT and POST contribute the largest positive net amounts, at 206,245.48 and 66,230.64, respectively**, demonstrating that postage-related charges can inflate reported merchandise revenue when included in transaction totals.
- **AMAZONFEE generates the largest negative net contribution (-221,520.50)**, followed by manual entries under stock code M (-68,674.19), highlighting the financial significance of administrative and accounting adjustments.
- **Other non-merchandise codes represent discounts, bank charges, commissions, carriage, samples, and bad-debt adjustments**, reinforcing the need to distinguish operational charges from product sales.
- **The relatively small overall net contribution conceals substantial offsetting positive and negative amounts**, meaning that evaluating non-merchandise transactions solely by their net percentage would understate their influence on revenue calculations.

**EDA Decision**

Exclude clearly identifiable non-merchandise charges and accounting adjustments from the eventual merchandise revenue target, subject to validating the transaction classifications. Investigate ambiguous manual entries separately before establishing final filtering rules.